# Proyecto Final

In [48]:
# Librerias estandar 
import pandas as pd 
import datasets 
import os 
from tqdm.notebook import tqdm
import json
import numpy as np

# LLM's 
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_ollama import OllamaLLM

**Ejercicio 1**

Utilizando AG News, se deberán implementar y comparar varios sistemas y enfoques vistos a lo largo de la 
asignatura. En este ejercicio, la entrada del sistema deberá ser únicamente la columna **description**. La 
columna **title** **no** deberá utilizarse como parte del texto de entrada para clasificar.

### Datos 

In [6]:
dataset = datasets.load_dataset("sh0416/ag_news")
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

In [7]:
train = pd.DataFrame(dataset["train"])
test = pd.DataFrame(dataset["test"])

print(f"Dimensiones train: {train.shape}")
print(f"Dimensiones test: {test.shape}")

print(f"\nTrain:")
train.sample(5)

Dimensiones train: (120000, 3)
Dimensiones test: (7600, 3)

Train:


,label,title,description
48061,3,Lehman eyes UK hedge fund,Source says the investment bank is in talks to...
72644,3,C amp;W to sell Japanese arm to Softbank,Cable amp; Wireless has agreed to sell its Ja...
24235,2,Miami Beats Florida St. 16-10 in OT (AP),"AP - New season, new conference, same result. ..."
109320,2,Cubs sign shortstop,CHICAGO The Chicago Cubs have come to terms on...
63499,4,Panel Recommends Sharp Cuts in Fish Stocks (AP),AP - Catches of cod and other fish stocks must...


**Ejercicio 2**: Sistema de Recuperación de Información de tipo RAG y Agentes

En este ejercicio se deberá construir un sistema de recuperación de información de tipo RAG (Retrieval-Augmented Generation) y utilizar la información recuperada para realizar una tarea de clasificación. 

El contenido de la columna `title` se utilizará como consulta (query). El sistema deberá recuperar el contenido de la columna `description` más relevante y, a partir de la información recuperada, predecir la categoría correspondiente al contenido de `title` analizado.

Para esta tarea se deberán implementar y comparar dos sistemas:
1. Un sistema básico basado en RAG.
2. Un sistema basado en RAG ampliado con un agente sencillo.

---

**1. Sistema Básico Basado en RAG**
El sistema basado en RAG debe incluir, como mínimo:
* **Definición de la unidad de recuperación:** Estructuración de los documentos indexables.
* **Embeddings:** Generación de representaciones vectoriales para el texto.
* **Indexación vectorial:** Creación de una base de datos vectorial para almacenar y buscar los embeddings.
* **Recuperación top-k:** Extracción de los $k$ fragmentos de `description` más similares a la consulta (`title`).
* **Predicción de la categoría con un LLM:** Uso del modelo de lenguaje para emitir el veredicto final basándose en el contexto recuperado.

---

**2. Sistema Avanzado Basado en Agente**
El sistema con agente debe incorporar una función de control que añada lógica y flexibilidad al flujo tradicional de RAG. Ejemplos de funciones de control a implementar:
* **Decidir si recuperar información o no:** Evaluar si la consulta requiere contexto externo o si puede responderse directamente.
* **Reformular la consulta realizada:** Optimizar o expandir el `title` original para mejorar la calidad de la búsqueda en la base de datos vectorial.
* **Seleccionar entre varias estrategias de recuperación:** Alternar dinámicamente entre diferentes métodos de búsqueda o valores de $k$.
* **Invocar una tool externa de clasificación:** En particular, podrá utilizar una tool que invoque el modelo de clasificación entrenado en el primer ejercicio (por ejemplo, `DistilRoBERTa_Transformer`) para predecir la categoría final a partir del contenido recuperado.
* **Controlar el flujo de ejecución:** Decidir los pasos secuenciales o condicionales que debe seguir el sistema según las respuestas intermedias.

## 1. RAG Basico

### Unidad de recuperacion

En primer lugar, vamos a definir la unidad de recuperación. En este caso esta especificado que el corpus del que se quiere realizar la recuperación son los fragmentos de texto de la columna *description*. Debido a que se pueden cargar a Chroma los documentos directamente desde un dataframe de pandas, vamos a crear dicho dataframe. 

In [8]:
# Definimos los documentos mediante el objeto Document de langchain
retrieval_units = [Document(page_content=row["description"],
                            metadata={
                                "title": row["title"],
                                "original_label": int(row["label"])
                            })
                   for _, row in train.iterrows()
]


print(f"Hay {len(retrieval_units)} documentos",
      f"\nEjemplos:")
retrieval_units[0:5]

Hay 120000 documentos 
Ejemplos:


[Document(metadata={'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'original_label': 3}, page_content="Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."),
 Document(metadata={'title': 'Carlyle Looks Toward Commercial Aerospace (Reuters)', 'original_label': 3}, page_content='Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.'),
 Document(metadata={'title': "Oil and Economy Cloud Stocks' Outlook (Reuters)", 'original_label': 3}, page_content='Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.'),
 Document(metadata={'title': 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters)', 'original_label': 3}, page_content='Reuter

### Embeddings 

Puesto que se va a utilizar gemma2 com Ollama, vamos a utilizar el modelo de embeddings correspondiente.

In [9]:
# Modelo de embedding de gemma2
embeddings = OllamaEmbeddings(model="gemma2")

embeddings

OllamaEmbeddings(model='gemma2', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

### Creacion de base de datos vectorial

Una vez esta definido el modelo de embeddings, ya se puede crear la base de datos vectorial con Chroma.

In [10]:
import os
from langchain_community.vectorstores import Chroma
from tqdm.notebook import tqdm

DB_DIR = "./chroma_DB"
BATCH_SIZE = 500  # Subir de 500 en 500 documentos para no saturar Ollama

# Comprobar si la base de datos ya existe en el disco local
if os.path.exists(DB_DIR):
    print("Cargando la base de datos vectorial existente desde el disco...")
    vector_DB = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    print("¡Base de datos cargada con éxito!")
else:
    print("La base de datos no existe. Iniciando proceso de indexación...")

    # Inicializar la base de datos vacía con el primer lote
    vector_DB = Chroma.from_documents(
        documents=retrieval_units[:BATCH_SIZE],
        embedding=embeddings,
        persist_directory=DB_DIR,
    )

    # Agregar el resto de los documentos en lotes con barra de progreso
    for i in tqdm(
        range(BATCH_SIZE, len(retrieval_units), BATCH_SIZE),
        desc="Indexando documentos en Chroma",
    ):
        chunk = retrieval_units[i : i + BATCH_SIZE]
        vector_DB.add_documents(chunk)

    print("¡Indexación completada con éxito!")

Cargando la base de datos vectorial existente desde el disco...


/tmp/ipykernel_1027645/1670653636.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_DB = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)


¡Base de datos cargada con éxito!


### Recuperación top-k 

La base de datos vectorial ha sido creada y los documentos indexados. Ahora toca probar a realizar una recuperación top-k. Para esta tarea se va a utilizar un enfoque hibrido. 

Por un lado, se se van a buscar las relaciones semánticas calculando la similaridad coseno entre los embeddings. La ventaja de este enfoque es que captura el concepto general, pero no puede enfocarse en palabras clave.

Por otro lado, se va a utilizar el algoritmo BM25, una evolución del TF-IDF. Si bien no entiende el contexto, si hay palabras clave las identifica. Este algoritmo, a diferencia de TF-IDF, hace que la importancia de una palabra en un texto no crezca linealmente con su frecuencia, sino que cuanto más aparece menos aporta cada aparición (es como decir que, a partir de cierto numero de apariciones, ya ha quedado claro que es importante, no va a escalar la importacia hasta el infinito). También penaliza la frecuencia de las palabras en todos los documentos. Por último, cabe destacar que normaliza por el tamaño de cada documento, mirando la "densidad". La expresión matemática es: 

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Para combinar los resultados (puesto que ambos métodos tienen escalas distintas) se va a utilizar el algoritmo RRF, el cual calcula una métrica en funcion de los rakings de cada método, y esta ya se puede hacer como un último ranking.

$$RRF\_Score(d \in D) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$

In [ ]:
# Configurar el retrieval denso
chroma_retriever = vector_DB.as_retriever(search_kwargs={"k": 10})

# Configurar el retrieval disperso
bm25_retriever = BM25Retriever.from_documents(retrieval_units)
bm25_retriever.k = 10

# Algoritmo RRF 
def get_hybrid_top_k(query, k=3, rrf_constant=60):
    
    # Recuperamos los 10 primeros candidatos de cada método
    dense_docs = chroma_retriever.invoke(query)
    sparse_docs = bm25_retriever.invoke(query)

    rrf_scores = {}

    # Aplicar RRF al canal denso
    for rank, doc in enumerate(dense_docs):
        doc_id = doc.page_content
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {"doc": doc, "score": 0.0}
        rrf_scores[doc_id]["score"] += 1.0 / (rrf_constant + (rank + 1))

    # Aplicar RRF al canal disperso
    for rank, doc in enumerate(sparse_docs):
        doc_id = doc.page_content
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {"doc": doc, "score": 0.0}
        rrf_scores[doc_id]["score"] += 1.0 / (rrf_constant + (rank + 1))

    # Ordenar de mayor a menor la métrica hibrida
    sorted_items = sorted(
        rrf_scores.values(), key=lambda x: x["score"], reverse=True
    )

    # Devolver el top-k
    return [item["doc"] for item in sorted_items[:k]]

In [24]:
# Cogemos un titulo de test al azar
example_title = test["title"][10]

# Realizamos el retrieval
docs_recuperados = get_hybrid_top_k(example_title, k=3)

print(F"Titulo buscado: {example_title}\nTextos recuperados:")
for doc in docs_recuperados:
    print(doc.page_content)

Titulo buscado: Group to Propose New High-Speed Wireless Format
Textos recuperados:
Will launch a fixed wireless service in a 'major city' next year
Electronic document giant Adobe Systems says it will partner with digital certificate company GeoTrust to provide technology to allow documents that use Adobe #39;s popular Portable Document Format to be digitally certified.
After some years on the drawing board, Japanese mobile giant NTT DoCoMo has announced its work on a common platform for 3G phones has come to fruition.


Como se puede apreciar, el titulo esta relacionado con tech y todos los textos recuperados son de tecnologia. Se va a probar a recuperar un titulo de train a ver si encuentra su descripción original.

In [36]:
# Cogemos el titulo y descripcion de train
title = train["title"][1]

print(F"Titulo original:\n - {title}\n")

# Realizamos el retrieval
docs_recuperados = get_hybrid_top_k(title, k=10)
print(f"Titulos de documentos recuperados")
for doc in docs_recuperados:
    print(doc.metadata["title"])

Titulo original:
 - Carlyle Looks Toward Commercial Aerospace (Reuters)

Titulos de documentos recuperados
ArvinMeritor posts \$153 million loss in quarter after unit sale
Dimon to Acquire tobacco Leaf Rival Standard Commercial
Genesis Crash Blamed on Installation Error
Delta lands \$500M in financing
News: New Study to Investigate Demise of Coral Reef Ecosystems
Carlyle Increases Its Interest in Debt Deals
Microsoft Puts New Life Into Its Systems Management
Navistar Results Rise on Truck Demand
Google #39;s debut in the stock market sends a mixed signal
Symantec fires up firewall appliance for smaller firms


Si bien no se llega a recuperar el titulo original, se recuperan muchos muy similaraes. Al final hay que tener en cuenta que, por mucho que los titulos se relacionen con su contenido, tambien un titulo puede tener mucha relacion con otra descripcion, sobre todo si tratan temas muy similares. Cabe destacar la recuperacion de un titulo que tambien habla de "Carlyle", probablemente el algoritmo BM25 ha ayudado aqui, ya que es una palabra clave del titulo query.

### Predicción con LLM

Por último de esta primera parte, se va a realizar la predicción de textos via promting y RAG. La idea es escoger un LLM, crear un buen promt para esta tarea y añadirle los documentos recuperados utilizando del title com query. 

In [75]:
# Modelo LLM
llm = OllamaLLM(model="gemma2", temperature=0.0)

# Definir el promt 
PROMPT_TEMPLATE = """
Mira el siguiente contexto recuperado de una base de datos de noticias y clasifica el título proporcionado en una de estas 4 categorías numéricas:
1: World (Mundo)
2: Sports (Deportes)
3: Business (Negocios)
4: Sci/Tech (Ciencia/Tecnología)

Contexto relevante:
{context}

Título a clasificar:
{title}

Responde estrictamente con un único número entero (1, 2, 3 o 4) que represente la categoría. No añadas explicaciones, introducciones ni texto adicional.
Respuesta:"""

# Listas para almacenar las etiquetas reales y las predicciones del modelo
y_true = []
y_pred = []

# Bucle de evaluación acotado a las primeras 100 muestras de test

muestras_evaluacion = test.sample(n=500, random_state=123)

for idx, row in tqdm(
    muestras_evaluacion.iterrows(),
    total=500,
    desc="Evaluando RAG con Gemma2"):
    
    # Sacamos el titulo y la etiqueta real
    titulo_query = row["title"]
    etiqueta_real = int(row["label"])

    # Recuperación híbrida del Top-K utilizando la función personalizada
    docs_recuperados = get_hybrid_top_k(titulo_query, k=5)

    # Concatenar el contenido de los documentos recuperados para el contexto
    contexto_formateado = "\n".join(
        [f"- {doc.page_content}" for doc in docs_recuperados]
    )

    # Construir el prompt final inyectando las variables
    prompt_final = PROMPT_TEMPLATE.format(
        context=contexto_formateado, title=titulo_query
    )

    # Paso D: Invocar al LLM y registrar los resultados
    respuesta_llm = llm.invoke(prompt_final).strip()

    # Guardar los datos para la evaluación analítica posterior
    y_true.append(etiqueta_real)
    y_pred.append(int(respuesta_llm))
    
y_true = np.array(y_true)
y_pred = np.array(y_pred)
print("¡Evaluación de las 100 predicciones completada!")

Evaluando RAG con Gemma2:   0%|          | 0/500 [00:00<?, ?it/s]

¡Evaluación de las 100 predicciones completada!


In [76]:
print(f"Accuracy obtenido: {np.array(y_true == y_pred).sum() / len(y_true)}") #type:ignore

Accuracy obtenido: 0.77


El accuracy obtenido es decente. Sin embargo, si lo comparamos con los métodos del ejercicio 1, se queda bastante atras. Se llega un poco a la misma conclusion, un método genérico obtendrá resultados más mediócres que uno específico. 

Sin embargo, el objetivo al final era implementar en conjunto el RAG con un LLM para clasificación, lo cual se ha conseguido. 

## 2. RAG avanzado con Agente